# 03 Trees and ensembles

One tree and why it overfits, two ways to prune it, then bagging and random forests (less variance) and boosting (less bias), all compared with matrix pricing on valid.

Class notebooks followed: `RTrees_boston` (steps 2 to 5), `Regression_Improve Bagg & RF with Boosting` (steps 6, 8 to 10), `RF_risk` (step 7). Changes for our data: time-ordered folds instead of `cv=5`, searches on a 30% sample of the training months, and the shared preprocessing from `ml2.features`.

Validation months only: the test months are scored once, in `99_final`. Runs top to bottom in about 15 minutes on a laptop; figures go to
`reports/figures/03_*.png`.

In [ ]:
# Deck style for every figure: IG is always teal, HY always orange
import matplotlib.pyplot as plt

DECK = {"teal": "#114A6B", "orange": "#FF5F00", "slate": "#3A4550", "grey": "#8A949C", "pale": "#DCE6EC"}
plt.rcParams.update({
    "font.family": "Arial", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": DECK["grey"], "axes.labelcolor": DECK["slate"], "axes.titleweight": "bold",
    "axes.titlecolor": DECK["teal"],
    "axes.prop_cycle": plt.cycler(color=[DECK["teal"], DECK["orange"], DECK["grey"], DECK["slate"]]),
    "xtick.color": DECK["slate"], "ytick.color": DECK["slate"], "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True, "grid.color": DECK["pale"],
    "figure.figsize": (10, 5.6), "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## 1. Setup

Trees need no scaling: `make_preprocessor(scale=False, one_hot=False)` imputes and codes the categories as integers.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline

from ml2.baseline import matrix_same_date
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import improvement_ci, metrics_by_stratum, regression_metrics
from ml2.features import make_preprocessor, xy
from ml2.splits import issuer_holdout_folds, walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["target"]
PREFIX = "03"
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Save a figure to reports/figures/<prefix>_<name>.png."""
    fig.savefig(FIG / f"{PREFIX}_{name}.png", dpi=150, bbox_inches="tight")


check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train)
X_va, y_va = xy(valid)

# Searches run on a 30% sample of train (time order kept); chosen models are refitted on all of train
sub = train.sample(frac=0.3, random_state=42).sort_values("snapshot_date")
X_s, y_s = xy(sub)
cv = walk_forward_folds(sub.loc[X_s.index, "snapshot_date"])  # replaces cv=5 / KFold in the class notebooks
# New-issuer question: in each time-ordered fold, a fifth of the companies are held out (see ml2.splits)
new_issuer_folds = issuer_holdout_folds(sub.loc[X_s.index, "snapshot_date"], sub.loc[X_s.index, "issuer_ticker"])


def new_issuer_cost(model):
    """MAE on the held-out companies, trained with and without them (same number of rows, same months)."""
    seen, unseen = [], []
    for tr_seen, tr_unseen, held in new_issuer_folds:
        X_h, y_h = X_s.iloc[held], y_s.iloc[held]
        seen.append(mean_absolute_error(y_h, clone(model).fit(X_s.iloc[tr_seen], y_s.iloc[tr_seen]).predict(X_h)))
        unseen.append(mean_absolute_error(
            y_h, clone(model).fit(X_s.iloc[tr_unseen], y_s.iloc[tr_unseen]).predict(X_h)))
    return float(np.mean(seen)), float(np.mean(unseen))
print(f"train {X_tr.shape}, valid {X_va.shape}, search sample {X_s.shape}, {len(cv)} time-ordered folds")

import lightgbm as lgb
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeRegressor, plot_tree


def tree_pipe(est):
    return Pipeline([("pre", make_preprocessor(scale=False, one_hot=False)), ("est", est)])

In [ ]:
# The bar: matrix pricing re-marked on each date (the gate baseline), on valid. About 1.5 minutes.
valid = valid.copy()
valid["pred_matrix_same_date"] = matrix_same_date(valid)
results = {}  # model name -> metrics by grade on valid, for the comparison table


def vs_bar(pred, name):
    """MAE by grade for predictions on X_va's rows, stored in `results`, shown next to the gate baseline."""
    rows = valid.loc[X_va.index].assign(pred=np.asarray(pred))
    results[name] = metrics_by_stratum(rows, TARGET, "pred", "grade_bucket").set_index("stratum")
    bar = metrics_by_stratum(rows, TARGET, "pred_matrix_same_date", "grade_bucket").set_index("stratum")
    out = pd.DataFrame({name: results[name]["mae_bps"], "matrix re-marked (bar)": bar["mae_bps"]}).T
    out["IG gain"] = 1 - out["IG"] / out.loc["matrix re-marked (bar)", "IG"]
    out["HY gain"] = 1 - out["HY"] / out.loc["matrix re-marked (bar)", "HY"]
    return out[["IG", "HY", "ALL", "IG gain", "HY gain"]].round(3)


vs_bar(valid.loc[X_va.index, "pred_matrix_same_date"], "Matrix pricing re-marked each date (bar)")

## 2. Initial tree

RTrees_boston, "Single Tree". A regression tree splits on the feature and threshold that most reduce the squared
error of the two halves, and predicts the mean spread of each leaf. Grown without limits it memorizes the training
rows.

In [ ]:
my_tree = tree_pipe(DecisionTreeRegressor(random_state=42)).fit(X_s, y_s)
train_mae = mean_absolute_error(y_s, my_tree.predict(X_s))
cv_mae = -cross_val_score(my_tree, X_s, y_s, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1).mean()
print(f"train MAE {train_mae:.1f} bp, CV MAE {cv_mae:.1f} bp, {my_tree['est'].get_n_leaves():,} leaves")

**Reading.** Grown without limits, the tree ends with 35,837 leaves for 35,837 training rows: one leaf per bond-month. It reproduces every training spread exactly (train MAE 0.0 bp), but on the later months of the time-ordered folds its error is 29.8 bp. That gap is overfitting. The tree has memorised the noise of the months it saw instead of learning rules that hold in later months: almost no bias, very high variance. Pruning (next) and averaging many trees (bagging, forests) are the two ways to fix it.

## 3. Reducing the complexity

RTrees_boston: grid search over depth and leaf size, scored by MAE on the time-ordered folds.

In [ ]:
tree_pruned_cv = GridSearchCV(tree_pipe(DecisionTreeRegressor(random_state=42)),
                              {"est__max_depth": range(4, 16, 2), "est__min_samples_leaf": [20, 50, 100, 200]},
                              scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1).fit(X_s, y_s)
tree_pruned = tree_pruned_cv.best_estimator_
print("best:", tree_pruned_cv.best_params_, f"CV MAE {-tree_pruned_cv.best_score_:.1f} bp")

res = pd.DataFrame(tree_pruned_cv.cv_results_)
grid_mae = -res.pivot(index="param_est__max_depth", columns="param_est__min_samples_leaf", values="mean_test_score")
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(grid_mae.values, cmap="viridis_r")
ax.set_xticks(range(grid_mae.shape[1]), grid_mae.columns); ax.set_yticks(range(grid_mae.shape[0]), grid_mae.index)
for i in range(grid_mae.shape[0]):
    for j in range(grid_mae.shape[1]):
        ax.text(j, i, f"{grid_mae.values[i, j]:.1f}", ha="center", va="center", color="white", fontsize=8)
ax.set_xlabel("min_samples_leaf"); ax.set_ylabel("max_depth"); ax.set_title("CV MAE (bp) of the tree grid")
fig.colorbar(im, ax=ax, label="bp")
save(fig, "tree_grid")

## 4. Cost-complexity pruning, 1-SE rule

RTrees_boston. The pruning path of the full tree gives the alphas at which leaves are cut. Our path has tens of
thousands of alphas (one per weakest link of a 34,000-leaf tree), so 25 of them spread over its upper range are
cross-validated. The 1-SE rule then picks the simplest tree whose CV error is within one standard error of the best.

In [ ]:
Z_s = my_tree["pre"].transform(X_s)
path = my_tree["est"].cost_complexity_pruning_path(Z_s, y_s)
alphas = np.unique(np.quantile(np.maximum(path.ccp_alphas[:-1], 0), np.linspace(0.90, 0.9999, 25)))
means, ses = [], []
for a in alphas:
    s = cross_val_score(tree_pipe(DecisionTreeRegressor(random_state=42, ccp_alpha=a)), X_s, y_s,
                        cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    means.append(s.mean()); ses.append(s.std() / np.sqrt(len(cv)))
means, ses = np.array(means), np.array(ses)
best = int(np.argmax(means))
alpha_1se = alphas[np.where(means >= means[best] - ses[best])[0][-1]]
tree_pruned_ccp = tree_pipe(DecisionTreeRegressor(random_state=42, ccp_alpha=alpha_1se)).fit(X_s, y_s)

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(alphas, -means, yerr=ses, marker="o", ms=3)
ax.axvline(alphas[best], color="grey", ls=":", label="best alpha")
ax.axvline(alpha_1se, color="red", ls="--", label="1-SE alpha")
ax.set_xscale("log"); ax.set_xlabel("ccp_alpha"); ax.set_ylabel("CV MAE (bp)")
ax.set_title("Cost-complexity pruning"); ax.legend()
save(fig, "ccp_1se")
print(f"best alpha {alphas[best]:.3g}, 1-SE alpha {alpha_1se:.3g}, {tree_pruned_ccp['est'].get_n_leaves()} leaves")

## 5. Comparison between the 3 trees

Same folds for every tree.

In [ ]:
rows = []
for name, model in {"Initial": my_tree, "Pruned (depth/leaf)": tree_pruned,
                    "Pruned (CCP 1-SE)": tree_pruned_ccp}.items():
    r = cross_validate(model, X_s, y_s, cv=cv, scoring={"mae": "neg_mean_absolute_error"}, n_jobs=-1)
    rows.append({"model": name, "CV MAE": -r["test_mae"].mean(), "leaves": model["est"].get_n_leaves()})
pd.DataFrame(rows).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(tree_pruned["est"], max_depth=2, feature_names=list(tree_pruned["pre"].get_feature_names_out()),
          filled=True, fontsize=8, ax=ax)
ax.set_title("Top of the pruned tree (depth 2)")
save(fig, "pruned_tree")

In [ ]:
best_tree = tree_pipe(DecisionTreeRegressor(random_state=42, **{k.split("__")[1]: v for k, v in
                                                                tree_pruned_cv.best_params_.items()}))
best_tree.fit(X_tr, y_tr)
vs_bar(best_tree.predict(X_va), "Pruned regression tree")

**Reading.** Both kinds of pruning cut the tree from 35,837 leaves to a few hundred and lower the CV error from 29.8 bp to about 27 bp. We keep the depth/leaf tree (depth 14, at least 50 bonds per leaf). It has the lowest CV error (27.0 bp) and is the one scored on valid (19.1 / 81.7 bp, 10% and 8% better than the bar). The cost-complexity tree with the 1-SE rule is almost as good (27.3 bp) with a third fewer leaves (336). Here the 1-SE rule picked the same alpha as the best one (0.946), so a simpler tree would have cost accuracy.

The first splits follow the order a credit analyst would use:

- **Rating first:** HY bonds average 255 bp against 88 bp for IG.
- **Among HY bonds, stock volatility:** companies whose stock moves more than 67% a year (106 bond-months) average 870 bp, the distressed names. The market prices default risk through the stock.
- **Among IG bonds, time to maturity:** bonds with more than 6.3 years left pay 103 bp against 69 bp. Lenders want more for lending longer.
- **Then company size and leverage:** bigger, less indebted companies pay less.

## 6. Bagging

Regression_Improve: a random forest with `max_features=None` is bagging of trees (every split may use every
feature). The out-of-bag (OOB) error uses, for each row, only the trees that did not see it.

In [ ]:
base = tree_pipe(RandomForestRegressor(n_estimators=100, max_features=None, oob_score=True, random_state=42))
cvBagg = GridSearchCV(base, {"est__max_depth": [None, 10], "est__min_samples_leaf": [10, 30]},
                      scoring="neg_mean_absolute_error", cv=cv, n_jobs=-2).fit(X_s, y_s)
print("bagging:", cvBagg.best_params_, f"CV MAE {-cvBagg.best_score_:.1f} bp")

bagg = tree_pipe(RandomForestRegressor(n_estimators=100, max_features=None, oob_score=True, random_state=42,
                                       n_jobs=-1, **{k.split("__")[1]: v for k, v in cvBagg.best_params_.items()}))
bagg.fit(X_tr, y_tr)
oob_mae = mean_absolute_error(y_tr, bagg["est"].oob_prediction_)
print(f"OOB MAE {oob_mae:.1f} bp against valid MAE {mean_absolute_error(y_va, bagg.predict(X_va)):.1f} bp")
vs_bar(bagg.predict(X_va), "Bagging (random forest, max_features=None)")

**Reading.** The out-of-bag error (13.0 bp) looks far better than the error on the valid months (19.3 bp). It's optimistic because each bond appears in the training data every month. When a row is left out of a tree's bootstrap sample, the same bond's previous and next months are usually still in it, and spreads barely move from one month to the next. So the "unseen" row is close to a copy of rows the tree did see, and OOB measures how well the model remembers known bonds in known months. Valid asks a harder question: later months, after a one-month gap, with market conditions the model has not seen. With panel data like ours, OOB is a quick check on the number of trees, not an estimate of future error. We choose settings on the time-ordered folds and judge on valid.

## 7. How many trees do we need?

RF_risk: out-of-bag MAE as trees are added (`warm_start=True`), on all of train.

In [ ]:
gridRF = GridSearchCV(base, {"est__max_depth": [None], "est__min_samples_leaf": [10, 30],
                             "est__max_features": ["sqrt", 0.3, 0.5]},
                      scoring="neg_mean_absolute_error", cv=cv, n_jobs=-2).fit(X_s, y_s)
print("random forest:", gridRF.best_params_, f"CV MAE {-gridRF.best_score_:.1f} bp")

Z_tr = make_preprocessor(scale=False, one_hot=False).fit(X_tr).transform(X_tr)
rf_warm = RandomForestRegressor(n_estimators=25, warm_start=True, oob_score=True, n_jobs=-1, random_state=42,
                                min_samples_leaf=gridRF.best_params_["est__min_samples_leaf"],
                                max_features=gridRF.best_params_["est__max_features"])
n_trees, oob_curve = list(range(25, 201, 25)), []
for n in n_trees:
    rf_warm.set_params(n_estimators=n).fit(Z_tr, y_tr)
    oob_curve.append(mean_absolute_error(y_tr, rf_warm.oob_prediction_))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_trees, oob_curve, marker="o")
ax.set_xlabel("Number of trees"); ax.set_ylabel("OOB MAE (bp)"); ax.set_title("How many trees do we need?")
save(fig, "oob_curve")

## 8. Random forest

Same grid plus `max_features`: each split sees only a random subset of features, which decorrelates the trees.

In [ ]:
rf = tree_pipe(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1,
                                     **{k.split("__")[1]: v for k, v in gridRF.best_params_.items()}))
rf.fit(X_tr, y_tr)
vs_bar(rf.predict(X_va), "Random forest")

## 9. Boosting: XGBoost, then LightGBM

Regression_Improve. Boosting fits small trees one after another, each on the errors of the ones before; the
learning rate shrinks each tree's contribution, so a lower rate needs more trees. LightGBM grows leaf-wise and is
limited by `num_leaves`; XGBoost grows level-wise and is limited by `max_depth`.

In [ ]:
gridXGB = GridSearchCV(tree_pipe(xgb.XGBRegressor(objective="reg:squarederror", n_estimators=500, subsample=0.8,
                                                  colsample_bytree=0.7, random_state=42, n_jobs=-1)),
                       {"est__max_depth": [5, 10], "est__learning_rate": [0.03, 0.05]},
                       scoring="neg_mean_absolute_error", cv=cv).fit(X_s, y_s)
print("XGBoost:", gridXGB.best_params_, f"CV MAE {-gridXGB.best_score_:.1f} bp")
gridLGBM = GridSearchCV(tree_pipe(lgb.LGBMRegressor(n_estimators=500, subsample=0.8, subsample_freq=1,
                                                    colsample_bytree=0.7, random_state=42, verbosity=-1)),
                        {"est__num_leaves": [20, 30], "est__max_depth": [5, 10],
                         "est__learning_rate": [gridXGB.best_params_["est__learning_rate"]]},
                        scoring="neg_mean_absolute_error", cv=cv).fit(X_s, y_s)
print("LightGBM:", gridLGBM.best_params_, f"CV MAE {-gridLGBM.best_score_:.1f} bp")

In [ ]:
best_xgb = gridXGB.best_estimator_.fit(X_tr, y_tr)
best_lgbm = gridLGBM.best_estimator_.fit(X_tr, y_tr)
vs_bar(best_xgb.predict(X_va), "XGBoost")

In [ ]:
vs_bar(best_lgbm.predict(X_va), "LightGBM")

## 10. Interpretation

Built-in importances (gain for XGBoost, split counts for LightGBM; from training, on the transformed columns, and
biased toward features with many split points), permutation importance on
valid (how much the valid error rises when one feature is shuffled: more trustworthy), and SHAP values (how much
each feature moves each prediction, and in which direction).

In [ ]:
best = min([best_xgb, best_lgbm], key=lambda m: mean_absolute_error(y_va, m.predict(X_va)))
names = list(best["pre"].get_feature_names_out())
impurity = pd.Series(best["est"].feature_importances_, index=names).sort_values(ascending=False)

sample = X_va.sample(3000, random_state=42)
# n_jobs=1: parallel permutation workers crash with XGBoost models on Windows
pi = permutation_importance(best, sample, y_va.loc[sample.index], scoring="neg_mean_absolute_error",
                            n_repeats=3, random_state=42, n_jobs=1)
perm = pd.Series(pi.importances_mean, index=sample.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
impurity.head(15)[::-1].plot.barh(ax=axes[0]); axes[0].set_title("Built-in importance (train)")
perm.head(15)[::-1].plot.barh(ax=axes[1])
axes[1].set_title("Permutation importance (valid)"); axes[1].set_xlabel("Rise in MAE when shuffled (bp)")
save(fig, "importances")

In [ ]:
Z_sample = best["pre"].transform(sample)
shap_values = shap.TreeExplainer(best["est"]).shap_values(Z_sample)
shap.summary_plot(shap_values, Z_sample, feature_names=names, show=False, max_display=15)
fig = plt.gcf(); fig.suptitle("SHAP values of the best boosting model, valid sample")
save(fig, "shap_summary")

**Reading.** Yes, the top features make financial sense. On valid, the error rises most when we shuffle:

- the rating bucket (+9.6 bp)
- years to maturity (+8.8 bp)
- the company's total debt and market cap (about +5 bp each)
- leverage (+3.6 bp)

The SHAP plot shows each one pushing in the expected direction:

- **HY bonds:** +100 to +300 bp.
- **Longer maturity and higher leverage:** wider spreads.
- **Bigger companies:** tighter spreads.
- **High stock volatility:** a few very wide spreads, the distressed names.
- **Coupon:** a higher coupon goes with a wider spread, because the coupon was set to match the company's risk when the bond was issued.

Three things are less expected:

- **Missing-data flags:** the built-in (gain) importance ranks the "missing distance to default / market cap / leverage" flags near the top. A company with no confirmed stock data is a signal in itself: it's often private or smaller. Gain importance is measured on train and favours such splits, so we rely on the permutation ranking.
- **Distance to default:** it ranks low. It is built from volatility and leverage, which the model already sees, so shuffling it alone costs little.
- **Market-wide spreads** (HY and BBB index spreads, the 2-year Treasury rate): they matter little. They are the same for every bond on a date, so they can only move the overall level from month to month, and two years of one rate regime give little variation to learn from.

## 11. Comparison table

Valid, every model against the gate baseline on the same rows.

In [ ]:
table = pd.concat({name: r["mae_bps"] for name, r in results.items()}, axis=1).T[["IG", "HY", "ALL"]]
bar = table.loc["Matrix pricing re-marked each date (bar)"]
table["IG gain vs bar"] = 1 - table["IG"] / bar["IG"]
table["HY gain vs bar"] = 1 - table["HY"] / bar["HY"]
extra = pd.DataFrame({name: r.loc["ALL", ["rmse_bps", "r2"]] for name, r in results.items()}).T
table = table.join(extra).round(3)
table.to_csv(FIG.parent / f"{PREFIX}_comparison_valid.csv")
table

### Lines for RUN_LOG.md

For `notebooks/RUN_LOG.md`; the parameters are printed in the last cell.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
for name, r in table.iterrows():
    print(f"| {today} | <your name> | 03_trees_ensembles | {name} | all 33 | fit on train, scored on valid | "
          f"{r['IG']:.1f} / {r['HY']:.1f} | gain vs bar IG {r['IG gain vs bar']:.1%}, "
          f"HY {r['HY gain vs bar']:.1%} |")

The best boosting model was picked on valid, so its gain on valid is a little optimistic; the gate itself is judged
once, on test, in `99_final`. Its 95% company-bootstrap interval on valid, as the gate will compute it:

In [ ]:
rows = valid.loc[X_va.index].assign(pred=best.predict(X_va))
pd.DataFrame({g: improvement_ci(rows[rows.grade_bucket == g], TARGET, "pred", "pred_matrix_same_date")
              for g in ["IG", "HY"]}).T.round(3)

## 12. New-issuer question

In each time-ordered fold, a fifth of the companies in the validation months are held out. The best model is
trained twice on the same number of rows from the same months: once with those companies' earlier bonds, once
without them. Both are scored on the held-out companies only, so the gap is the cost of a company never seen.

In [ ]:
best_params = (gridXGB if best is best_xgb else gridLGBM).best_estimator_
seen_mae, unseen_mae = new_issuer_cost(best_params)
print(f"Held-out companies: {seen_mae:.1f} bp when the model has seen them, {unseen_mae:.1f} bp when it has not")

**Reading.** For a company it has never seen, the model is about 10 bp worse (29.1 bp against 19.3 bp), roughly 50% more error. With a known company, the model has learned that company's usual spread level from its other bonds, including things our features don't capture, such as how the market sees its management or its industry niche. For a new issuer it has only the bond's terms, the stock and balance sheet, and the market backdrop. A pricing desk should therefore treat the model's price for a new issuer's first bond as a first estimate: compare it with the matrix price and similar bonds, and have a trader check it until the issuer's own bonds have traded for a few months. (These numbers come from the 30% search sample with smaller training sets, so compare them with each other, not with the valid MAE in the table.)

## 13. Chosen settings

The candidates for `99_final`.

In [ ]:
print("best boosting model:", "XGBoost" if best is best_xgb else "LightGBM")
print(best["est"])
print("pruned tree:", tree_pruned_cv.best_params_, "| bagging:", cvBagg.best_params_, "| forest:", gridRF.best_params_)